# Thesis Phase 3, 4, 5: The Modeling Race & XAI 🏎️
**Objective:** Train and Compare Baseline vs Hybrid Models.
**Environment:** Local Repository Execution (GPU supported if available).

In [ ]:
# 1. Setup & Imports (Auto-Install Dependencies)
import sys
import subprocess
import os

def install(package):
    print(f"[*] Installing {package}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# Check & Install critical libs
REQUIRED = ['gensim', 'shap', 'transformers', 'torch', 'scikit-learn']
for req in REQUIRED:
    try:
        __import__(req.replace('-', '_')) # 'scikit-learn' -> 'sklearn'
    except ImportError:
        install(req)

import pandas as pd
import numpy as np
import torch
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import f1_score, accuracy_score, top_k_accuracy_score
from gensim.models import Word2Vec
from transformers import DistilBertTokenizer, DistilBertModel
import shap
import matplotlib.pyplot as plt
from scipy.stats import entropy

# --- CONFIGURATION ---
BASE_DIR = r"E:\Github\repo-phaze7r\geospatial-tagging-thesis"
DATA_DIR = os.path.join(BASE_DIR, 'datareported')

print(f"[*] Base Directory: {BASE_DIR}")
print(f"[*] Data Directory: {DATA_DIR}")
   
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Computing Device: {device}")

In [ ]:
# 2. Load Datasets
try:
    df_base = pd.read_csv(os.path.join(DATA_DIR, 'dataset_baseline.csv')).fillna("")
    df_hyb = pd.read_csv(os.path.join(DATA_DIR, 'dataset_hybrid.csv')).fillna("")
    
    common_ids = set(df_base['osm_id']).intersection(set(df_hyb['osm_id']))
    if not common_ids:
        print("[!] Warning: No common IDs found between Baseline and Hybrid datasets.")
        print("    Check if Notebook 2 ran correctly.")
        
    df_base = df_base[df_base['osm_id'].isin(common_ids)].sort_values('osm_id')
    df_hyb = df_hyb[df_hyb['osm_id'].isin(common_ids)].sort_values('osm_id')
    y = df_base['city']
    print(f"[*] Loaded {len(df_base)} samples.")
except FileNotFoundError:
    print("[!] Datasets not found.")
    print("    Did you run Notebook 2?")

In [ ]:
# 3. Baseline Model
if len(df_base) > 0:
    print("[*] Training Baseline (Word2Vec)...")
    sentences = [t.split() for t in df_base['text_baseline']]
    # Check if we have sentences
    if not any(sentences):
        print("[!] Warning: Baseline text is empty. Skipping W2V.")
        hit5, f1_b = 0, 0
    else:
        w2v = Word2Vec(sentences, vector_size=100, window=5, min_count=1)
        def get_w2v(t): 
            v=[w for w in t.split() if w in w2v.wv]
            return np.mean([w2v.wv[w] for w in v],axis=0) if v else np.zeros(100)
        X_b = np.array([get_w2v(t) for t in df_base['text_baseline']])
        X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(X_b, y, test_size=0.2, stratify=y)
        clf_b = LogisticRegression(max_iter=1000).fit(X_tr_b, y_tr_b)
        hit5 = top_k_accuracy_score(y_te_b, clf_b.predict_proba(X_te_b), k=5, labels=clf_b.classes_)
        f1_b = f1_score(y_te_b, clf_b.predict(X_te_b), average='weighted')
        print(f"Baseline - Hit@5: {hit5:.4f}, F1: {f1_b:.4f}")
else:
    print("[!] No data to train Baseline.")
    hit5, f1_b = 0, 0

In [ ]:
# 4. Hybrid Model
if len(df_hyb) > 0:
    print("[*] Training Hybrid (DistilBERT)...")
    try:
        tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
        model = DistilBertModel.from_pretrained('distilbert-base-uncased').to(device)
        def get_bert(txts): 
            embs=[]
            for i in tqdm(range(0, len(txts), 64)):
                batch = txts[i:i+64]
                # Handle empty strings to avoid errors
                batch = [t if t.strip() else "empty" for t in batch]
                inputs=tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=128).to(device)
                with torch.no_grad(): embs.append(model(**inputs).last_hidden_state[:,0,:].cpu().numpy())
            return np.vstack(embs)
            
        X_h = get_bert(df_hyb['text_hybrid'].fillna("").tolist())
        X_tr_h, X_te_h, y_tr_h, y_te_h = train_test_split(X_h, y, test_size=0.2, stratify=y)
        clf_h = SGDClassifier(loss='log_loss', penalty='elasticnet').fit(X_tr_h, y_tr_h)
        hit3 = top_k_accuracy_score(y_te_h, clf_h.predict_proba(X_te_h), k=3, labels=clf_h.classes_)
        f1_h = f1_score(y_te_h, clf_h.predict(X_te_h), average='weighted')
        enn = np.mean(entropy(clf_h.predict_proba(X_te_h), axis=1))
        print(f"Hybrid - Hit@3: {hit3:.4f}, F1: {f1_h:.4f}, ENN: {enn:.4f}")
    except Exception as e:
        print(f"[!] Hybrid Training Failed: {e}")
        hit3, f1_h, enn = 0, 0, 0
else:
    print("[!] No data to train Hybrid.")
    hit3, f1_h, enn = 0, 0, 0

In [ ]:
# 5. Save & Explain
res = {'Metric':['SRT (Hit@k)','F1','ENN'], 'Baseline':[hit5,f1_b,'N/A'], 'Hybrid':[hit3,f1_h,enn]}
pd.DataFrame(res).to_csv(os.path.join(DATA_DIR, 'comparison_report.csv'), index=False)
pd.DataFrame(res).to_csv(os.path.join(DATA_DIR, 'thesis_metrics_table.csv'), index=False)
print("[+] Results saved.")

# SHAP
from sklearn.feature_extraction.text import CountVectorizer

print("[*] Generating SHAP Plot...")
# Valid Sample Check
sample_texts = df_hyb['text_hybrid'][:1000].fillna("").tolist()
valid_samples = [t for t in sample_texts if len(t.strip()) > 1]

if len(valid_samples) < 5:
    print("[!] Warning: Insufficient text data for SHAP analysis.")
    print("    Hybrid dataset might be empty. Please Re-Run Notebook 2.")
else:
    try:
        vec = CountVectorizer(max_features=20)
        X_s = pd.DataFrame(vec.fit_transform(valid_samples).toarray(), columns=vec.get_feature_names_out())
        if X_s.shape[1] > 0:
            # Use filtered 'y' corresponding to valid samples if needed, or just slice y matching X_s length for simple check
            # For robust mapping, strictly we should filter y same way, but for SHAP proxy training it's approximate
            y_shap = y[:len(valid_samples)]
            
            explainer = shap.LinearExplainer(LogisticRegression().fit(X_s, y_shap), X_s, feature_perturbation="interventional")
            plt.figure()
            shap.summary_plot(explainer.shap_values(X_s), X_s, show=False)
            plt.savefig(os.path.join(DATA_DIR, 'shap_explanation.png'))
            print("[+] SHAP Plot saved.")
        else:
            print("[!] CountVectorizer found no valid vocabulary.")
    except ValueError as e:
        print(f"[!] SHAP Generation encountered an error: {e}")